## MobileNetV2 (MicroFlow-friendly)

Outputs

- `models/mobilenetv2_quantized.tflite` quantized
- `compiled_models/mobilenetv2_quantized.mlir`

In [5]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import tensorflow as tf

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("TF_LITE_DISABLE_XNNPACK", "1")

try:
    tf.get_logger().setLevel("ERROR")
except Exception:
    pass

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
COMPILED_DIR = REPO_ROOT / "compiled_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
COMPILED_DIR.mkdir(parents=True, exist_ok=True)

OUT_TFLITE = MODELS_DIR / "mobilenetv2_quantized.tflite"
OUT_MLIR = COMPILED_DIR / "mobilenetv2_quantized.mlir"

print("OUT_TFLITE:", OUT_TFLITE)
print("OUT_MLIR:", OUT_MLIR)

OUT_TFLITE: /home/nathan/Documents/ariel-os-tiny-ml/models/mobilenetv2_quantized.tflite
OUT_MLIR: /home/nathan/Documents/ariel-os-tiny-ml/compiled_models/mobilenetv2_quantized.mlir


In [6]:
def conv_bn_relu6_free(x, filters, kernel, strides=1):
    return tf.keras.layers.Conv2D(
        filters,
        kernel,
        strides=strides,
        padding="same",
        activation="relu6",
        use_bias=True,
    )(x)

def inverted_residual_no_skip(x, out_ch, expand, stride):
    in_ch = int(x.shape[-1])
    mid_ch = int(in_ch * expand)
    if expand != 1:
        x = tf.keras.layers.Conv2D(mid_ch, 1, padding="same", activation="relu6", use_bias=True)(x)
    x = tf.keras.layers.DepthwiseConv2D(3, strides=stride, padding="same", activation="relu6", use_bias=True)(x)
    x = tf.keras.layers.Conv2D(out_ch, 1, padding="same", activation=None, use_bias=True)(x)
    return x

def build_mobilenetv2_microflow(num_classes=10) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(32, 32, 3), batch_size=1, name="input")
    x = conv_bn_relu6_free(inputs, 16, 3, strides=1)

    # MobileNetV2-like stages, but NO residual Add ops.
    x = inverted_residual_no_skip(x, out_ch=24, expand=2, stride=2)   # 16x16
    x = inverted_residual_no_skip(x, out_ch=24, expand=2, stride=1)

    x = inverted_residual_no_skip(x, out_ch=32, expand=3, stride=2)   # 8x8
    x = inverted_residual_no_skip(x, out_ch=32, expand=3, stride=1)
    x = inverted_residual_no_skip(x, out_ch=64, expand=3, stride=1)

    # Classifier head that avoids MEAN.
    logits_map = tf.keras.layers.Conv2D(num_classes, 1, padding="same", activation=None, use_bias=True, name="logits_conv")(x)
    pooled = tf.keras.layers.AveragePooling2D(pool_size=(8, 8), strides=(8, 8), padding="valid", name="avg_pool_logits")(logits_map)
    logits = tf.keras.layers.Reshape((num_classes,), name="logits")(pooled)
    outputs = tf.keras.layers.Softmax(name="softmax")(logits)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="mobilenetv2_microflow")

model = build_mobilenetv2_microflow()
model.summary()

Model: "mobilenetv2_microflow"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (1, 32, 32, 3)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (1, 32, 32, 16)        │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (1, 32, 32, 32)        │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_5              │ (1, 16, 16, 32)        │           320 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (1, 16, 16, 24)        │           792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (1, 16, 16, 48)        │         1,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_6              │ (1, 16, 16, 48)        │           480 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (1, 16, 16, 24)        │         1,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (1, 16, 16, 72)        │         1,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_7              │ (1, 8, 8, 72)          │           720 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (1, 8, 8, 32)          │         2,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (1, 8, 8, 96)          │         3,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_8              │ (1, 8, 8, 96)          │           960 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (1, 8, 8, 32)          │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (1, 8, 8, 96)          │         3,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_9              │ (1, 8, 8, 96)          │           960 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (1, 8, 8, 64)          │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits_conv (Conv2D)            │ (1, 8, 8, 10)          │           650 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ avg_pool_logits                 │ (1, 1, 1, 10)          │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Reshape)                │ (1, 10)                │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (1, 10)                │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,034 (109.51 KB)

 Trainable params: 28,034 (109.51 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
epochs = 2
batch_size = 128

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.squeeze().astype("int64")
y_test = y_test.squeeze().astype("int64")
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose=2,
)
print("Test accuracy:", model.evaluate(x_test, y_test, verbose=0)[1])

Epoch 1/2
352/352 - 9s - 26ms/step - accuracy: 0.1002 - loss: 2.3029 - val_accuracy: 0.1038 - val_loss: 2.3029
Epoch 2/2
352/352 - 2s - 7ms/step - accuracy: 0.0982 - loss: 2.3029 - val_accuracy: 0.0970 - val_loss: 2.3032
Test accuracy: 0.10000000149011612


In [8]:
def representative_data_gen():
    n = min(200, x_train.shape[0])
    for i in range(n):
        yield [x_train[i : i + 1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
converter.experimental_enable_resource_variables = True

for attr in ("experimental_new_quantizer", "_experimental_new_quantizer"):
    if hasattr(converter, attr):
        setattr(converter, attr, False)

tflite_bytes = converter.convert()
OUT_TFLITE.write_bytes(tflite_bytes)
print("Wrote:", OUT_TFLITE, "bytes=", OUT_TFLITE.stat().st_size)

interp = tf.lite.Interpreter(model_path=str(OUT_TFLITE), experimental_delegates=[])
interp.allocate_tensors()
ops = [op.get("op_name") for op in interp._get_ops_details()]
print("Ops:", ops)
print("Contains MEAN?", "MEAN" in ops)

bad_scales = []
for td in interp.get_tensor_details():
    qp = td.get("quantization_parameters") or {}
    scales = qp.get("scales")
    if hasattr(scales, "size") and scales.size:
        if (scales == 0).any():
            bad_scales.append((td.get("index"), td.get("name"), int(scales.size)))

if bad_scales:
    print("Warning: zero quant scales (first 20):")
    for idx, name, n in bad_scales[:20]:
        print(" -", idx, name, "scales=", n)
else:
    print("Quant scales: OK (no zeros)")

from iree.compiler import tf as tfc
import shutil

ARTIFACTS_DIR = REPO_ROOT / "build" / "mlir_artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
SAVED_MODEL_DIR = ARTIFACTS_DIR / "mobilenetv2_quantized_saved_model"

shutil.rmtree(SAVED_MODEL_DIR, ignore_errors=True)
model.export(str(SAVED_MODEL_DIR))

mlir_bytes = tfc.compile_saved_model(
    str(SAVED_MODEL_DIR),
    import_only=True,
    exported_names=["serve"],
)
OUT_MLIR.write_bytes(mlir_bytes)
print("Wrote MLIR:", OUT_MLIR, "bytes=", OUT_MLIR.stat().st_size)

Saved artifact at '/tmp/tmp48oc9c4_'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 32, 32, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 10), dtype=tf.float32, name=None)
Captures:
  136587883226768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136587883227920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136587883227728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136587883225808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136587883228496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136587883228112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136587883226384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136587883228688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136587883229072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136587883228880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136587883229456: TensorSpec(

/home/nathan/Documents/ariel-os-tiny-ml/building/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1774616007.560235   69611 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774616007.560245   69611 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-27 13:53:27.560355: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp48oc9c4_
2026-03-27 13:53:27.561451: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-27 13:53:27.561461: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp48oc9c4_
2026-03-27 13:53:27.572219: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-27 13:53:27.630794: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on Save

Wrote: /home/nathan/Documents/ariel-os-tiny-ml/models/mobilenetv2_quantized.tflite bytes= 70488


RuntimeError: unsupported scale value (0.000000) in channel 0 for INT8 tensor 50 in XNNPACK delegateunsupported datatype (INT8) of tensor 50 in XNNPACK delegateNode number 20 (TfLiteXNNPackDelegate) failed to prepare.